# File-based walkthrough — upload files, build `config.json`, run the CLI

This is the **file-based** counterpart to the in-memory notebook. It matches a
UI where the user uploads a **model file** and a **data file**, the toolbox
auto-detects what it can, asks for the rest, writes a `config.json`, and runs
the command-line entry point.

The inspection/answer/resolve steps are identical to the package flow — the
only differences are that the model and data come from **files on disk** and
the run happens through `config.json` + the CLI instead of an in-memory call.

## 1. The uploaded files

Two things arrive from the user:

- a **model file** — `.pkl` (scikit-learn / XGBoost), `.pt`/`.pth` (PyTorch),
  or `.keras`/`.h5` (TensorFlow);
- a **data file** — a `.csv` **with a header row**. The header is what lets the
  toolbox name the features and check them against the model, so the UI should
  require a header (or offer a "name your columns" step for headerless files).

Here we create example files to stand in for the upload.

In [ ]:
import os, json, pickle
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier

os.makedirs("uploads", exist_ok=True)
FEATURES = ["Wind_Speed", "Temp", "Humidity", "Pressure"]
X = np.random.RandomState(0).randn(200, 4)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

df = pd.DataFrame(X, columns=FEATURES)
df.to_csv("uploads/data.csv", index=False)                 # header row present
model = RandomForestClassifier(n_estimators=50, random_state=0).fit(df, y)
with open("uploads/model.pkl", "wb") as f:
    pickle.dump(model, f)
print("wrote uploads/model.pkl and uploads/data.csv")

## 2. Inspect the uploaded model + data

`inspect_model` accepts an in-memory object; for a file we load it first (the
UI backend does the same when a file is uploaded). We pass the DataFrame so
feature names come from its header.

In [ ]:
import ai_explainability as aie

with open("uploads/model.pkl", "rb") as f:
    model = pickle.load(f)
df = pd.read_csv("uploads/data.csv")

profile = aie.inspect_model(model, df)
print(profile.describe())

## 3. Answer the remaining fields

Same field specs as the package flow — render `profile.missing` as a form.
For the file-based CLI we also decide on disk outputs and an `output_dir`,
since the whole point of the file flow is to produce files.

In [ ]:
answers = {
    "target_index": 0,               # binary classifier -> single target
    "save_excel": True,              # file flow -> write outputs
    "generate_notebook": False,      # set True to also render the .ipynb report
    "output_dir": "output/",
}

## 4. Resolve and write `config.json`

`resolve_config` produces a plain dict. Because it holds only JSON-friendly
values (no model/data objects), it serialises directly to the `config.json`
the CLI reads. We add the file paths the CLI needs to locate the assets.

In [ ]:
config = aie.resolve_config(profile, answers)
# The CLI loads the model/data from disk, so add the paths:
config["model_path"] = "uploads/model.pkl"
config["dataset_path"] = "uploads/data.csv"

with open("config.json", "w") as f:
    json.dump(config, f, indent=2)
print(json.dumps(config, indent=2, default=str))

## 5. Run the toolbox from the config

Both of these do the same thing — pick whichever your UI shells out to:

```bash
ai-explainability --config config.json
```

or, from Python (no subprocess):

In [ ]:
aie.run(["--config", "config.json"])

## 6. Collect the outputs

The run writes its audit (and optional notebook) into `output_dir`.

In [ ]:
produced = sorted(os.listdir(config["output_dir"])) if os.path.isdir(config["output_dir"]) else []
print("outputs:", produced)

## Summary for the UI (file-based)

1. User uploads a **model file** and a **data file (with header)**.
2. Backend loads them and calls `inspect_model(model, df)`.
3. Show inferred fields; render `profile.missing` as a form; show warnings
   (especially a header/model column mismatch).
4. On submit -> `resolve_config`, then inject `model_path` / `dataset_path`,
   and (for the file flow) `save_excel` / `output_dir`.
5. Write `config.json`; run `ai-explainability --config config.json`.
6. Offer the files in `output_dir` as downloads.

The **only** differences from the in-memory flow are steps 1 (files vs objects)
and 5 (config.json + CLI vs `aie.explain`). The inspection and form are shared.